# Baselines — Statistical / Simple Imputation (Sensor Time Series)
This notebook is a sectioned version of `baselines/baseline-stat.py`.

**Assumptions**
- `../data.txt` exists (or `data.txt` if you run from the repo root).
- You have `numpy`, `pandas`, `scipy`, and `matplotlib` installed.

**Notes**
- This baseline notebook uses the same *random missingness mask* and forced boundary missingness as the script.
- It does **not** include the outlier-masking step used in the LSTM-VAE notebook unless you add it explicitly.

In [ ]:
# === Section: Imports ===
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt

In [ ]:
# === Section: Load Data ===
# If you run this notebook from the repo root, use 'data.txt'.
# If you run it with CWD = baselines/, use '../data.txt'.
data_path = 'data.txt'

col_names = ['date','time','epoch','moteid','temperature','humidity','light','voltage']
df = pd.read_csv(data_path, sep=r'\s+', names=col_names, header=None)

sensor_id = 1
df = df[df.moteid == sensor_id].sort_values(['date','time']).reset_index(drop=True)
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
df = df.set_index('datetime')

temperature_series = df['temperature'].astype(float)
data = temperature_series.values
N = len(data)
print(f'Loaded {N} readings for sensor {sensor_id}')

In [ ]:
# === Section: Missingness Mask (Random + Boundary) ===
# Use the same random mask as in the VAE work for fair comparison.
missing_rate = 0.3
np.random.seed(42)

mask = np.ones(N, dtype=bool)
missing_indices = np.random.choice(N, size=int(N * missing_rate), replace=False)
mask[missing_indices] = False

# Force missing values at the start and end
mask[:5] = False
mask[-5:] = False

print(f'Masked points: {(~mask).sum()} / {N} ({(~mask).mean()*100:.1f}%)')

In [ ]:
# === Section: Baseline Imputation Methods ===
# Evaluate only on masked (missing) indices.

missing_true = data[~mask]

# Baseline 1: Mean imputation
mean_value = data[mask].mean()
mean_imputed = data.copy()
mean_imputed[~mask] = mean_value
mean_mae = np.mean(np.abs(mean_imputed[~mask] - missing_true))
mean_rmse = np.sqrt(np.mean((mean_imputed[~mask] - missing_true) ** 2))

# Baseline 1b: Median imputation
median_value = np.median(data[mask])
median_imputed = data.copy()
median_imputed[~mask] = median_value
median_mae = np.mean(np.abs(median_imputed[~mask] - missing_true))
median_rmse = np.sqrt(np.mean((median_imputed[~mask] - missing_true) ** 2))

# Baseline 1c: Mode imputation (often not meaningful for continuous data)
mode_value = mode(data[mask], keepdims=True).mode[0]
mode_imputed = data.copy()
mode_imputed[~mask] = mode_value
mode_mae = np.mean(np.abs(mode_imputed[~mask] - missing_true))
mode_rmse = np.sqrt(np.mean((mode_imputed[~mask] - missing_true) ** 2))

# Traditional imputation methods (3)
# 1) Last Observation Carried Forward (LOCF)
locf_imputed = data.copy().astype(float)
locf_imputed[~mask] = np.nan
locf_imputed = pd.Series(locf_imputed, index=df.index).ffill().bfill().values
locf_mae = np.mean(np.abs(locf_imputed[~mask] - missing_true))
locf_rmse = np.sqrt(np.mean((locf_imputed[~mask] - missing_true) ** 2))

# 2) Last Observation Carried Backward (LOCB)
locb_imputed = data.copy().astype(float)
locb_imputed[~mask] = np.nan
locb_imputed = pd.Series(locb_imputed, index=df.index).bfill().ffill().values
locb_mae = np.mean(np.abs(locb_imputed[~mask] - missing_true))
locb_rmse = np.sqrt(np.mean((locb_imputed[~mask] - missing_true) ** 2))

# 3) Hot-deck imputation (random donor sampling from observed values)
rng = np.random.default_rng(42)
donor_pool = data[mask]
if donor_pool.size == 0:
    raise RuntimeError('No observed points available for hot-deck imputation.')
hotdeck_imputed = data.copy().astype(float)
hotdeck_imputed[~mask] = rng.choice(donor_pool, size=(~mask).sum(), replace=True)
hotdeck_mae = np.mean(np.abs(hotdeck_imputed[~mask] - missing_true))
hotdeck_rmse = np.sqrt(np.mean((hotdeck_imputed[~mask] - missing_true) ** 2))

# Baseline 2: Forward-fill (then back-fill for initial gaps)
ffill_imputed = data.copy().astype(float)
ffill_imputed[~mask] = np.nan
ffill_imputed = pd.Series(ffill_imputed, index=df.index).ffill().bfill().values
ffill_mae = np.mean(np.abs(ffill_imputed[~mask] - missing_true))
ffill_rmse = np.sqrt(np.mean((ffill_imputed[~mask] - missing_true) ** 2))

# Baseline 3: Linear interpolation (then fill edges)
interp_imputed = data.copy().astype(float)
interp_imputed[~mask] = np.nan
interp_imputed = (
    pd.Series(interp_imputed, index=df.index)
    .interpolate(method='linear')
    .bfill()
    .ffill()
    .values
 )
interp_mae = np.mean(np.abs(interp_imputed[~mask] - missing_true))
interp_rmse = np.sqrt(np.mean((interp_imputed[~mask] - missing_true) ** 2))

print(f'Mean Imputation        MAE: {mean_mae:.4f}, RMSE: {mean_rmse:.4f}')
print(f'Median Imputation      MAE: {median_mae:.4f}, RMSE: {median_rmse:.4f}')
print(f'Mode Imputation        MAE: {mode_mae:.4f}, RMSE: {mode_rmse:.4f}')
print(f'LOCF (Forward)         MAE: {locf_mae:.4f}, RMSE: {locf_rmse:.4f}')
print(f'LOCB (Backward)        MAE: {locb_mae:.4f}, RMSE: {locb_rmse:.4f}')
print(f'Hot-deck               MAE: {hotdeck_mae:.4f}, RMSE: {hotdeck_rmse:.4f}')
print(f'Forward Fill           MAE: {ffill_mae:.4f}, RMSE: {ffill_rmse:.4f}')
print(f'Linear Interpolation   MAE: {interp_mae:.4f}, RMSE: {interp_rmse:.4f}')

In [ ]:
# === Section: Visual Check (Optional) ===
# Plot a short segment to compare imputations against ground truth.

segment_start = 0
segment_len = 500
segment_end = min(N, segment_start + segment_len)
idx = df.index[segment_start:segment_end]

plt.figure(figsize=(12, 4))
plt.plot(idx, data[segment_start:segment_end], label='Ground truth', linewidth=1)

masked_segment = data.copy().astype(float)
masked_segment[~mask] = np.nan
plt.plot(idx, masked_segment[segment_start:segment_end], label='Observed (masked)', linewidth=1)

# Pick a couple of baselines to overlay (edit as needed)
plt.plot(idx, interp_imputed[segment_start:segment_end], label='Linear interp', linewidth=1, alpha=0.9)
plt.plot(idx, locf_imputed[segment_start:segment_end], label='LOCF', linewidth=1, alpha=0.9)
plt.plot(idx, locb_imputed[segment_start:segment_end], label='LOCB', linewidth=1, alpha=0.9)

plt.title(f'Baselines on sensor {sensor_id} (segment {segment_start}:{segment_end})')
plt.xlabel('Datetime')
plt.ylabel('Temperature')
plt.legend()
plt.tight_layout()
plt.show()